
# 형태소 분석 기반 토큰화의 문제
- 형태소 분석기는 작성된 알고리즘 또는 학습된 내용을 바탕으로 토큰화를 하기 때문에 오탈자나 띄어쓰기 실수, 신조어, 외래어, 고유어 등이 사용된 경우 제대로 토큰화 하지 못한다.
- 그래서 발생 할 수있는 잠재적 문제점
    - 어휘사전을 크게 만든다.
        - 같은 의미의 단어가 형태소 분석이 안되어 여러개 등록될 수있다.
        - ex) 신조어 `돈쭐` 이라는 단어를 인식 못할 경우 `"돈쭐내러", "돈쭐나", "돈쭐냄"` 등이 다 등록 될 수 있다.
    - OOV(Out Of Vocab)에 대응하기 어렵게 만든다.
        - 같은 어근의 단어가 있지만 조사등이 바뀐 신조어등을 OOV로 인식할 수있다.




> ### 어휘 사전(Vocabulary)과 Out Of Vocabulary (OOV)
> 
> - 언어 모델링에서 **어휘 사전(Vocabulary)**은 모델이 처리할 수 있는 단어(토큰)들의 집합이다.  
> - 어휘 사전은 보통 전체 데이터셋을 토큰화한 후, 각 토큰을 고유한 정수 인덱스로 매핑해 만든다.
>    - 매핑된 정수는 모델에 입력되는 텍스트 데이터를 숫자 형식으로 변환해 모델이 처리할 수 있도록 돕는다.
>    - 예시) {"I": 1, "he": 2, "you": 3, ...}
> - **Out Of Vocabulary (OOV)**
>    - 어휘 사전(Vocab): 코퍼스를 구성하는 모든 토큰의 집합.
>    - **OOV**란 어휘 사전에 포함되지 않은 토큰을 의미하며, 모델이 해당 토큰을 처리할 수 없기 때문에 일반적으로 특별한 토큰(예: `[UNK]`)으로 대체되거나 다른 방식으로 처리된다.



In [ ]:
# %pip install korpora tokenizers

  Using cached xlrd-2.0.1-py2.py3-none-any.whl.metadata (3.4 kB)
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ------------------------------ --------- 1.8/2.4 MB 8.4 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 9.3 MB/s eta 0:00:00
Using cached xlrd-2.0.1-py2.py3-none-any.whl (96 kB)

   -------- ------------------------------- 1/5 [xlrd]
   ---------------- ----------------------- 2/5 [korpora]
   ---------------- ----------------------- 2/5 [korpora]
   ------------------------ --------------- 3/5 [huggingface-hub]
   ------------------------ --------------- 3/5 [huggingface-hub]
   ------------------------ --------------- 3/5 [huggingface-hub]
   ---------------------------------------- 5/5 [tokenizers]

Note: you may need to restart the kernel to use updated packages.


# Subword Tokenization(하위 단어 토큰화)

## 정의

- Subword Tokenization은 단어를 더 작은 단위(subword)로 나누어 텍스트를 토큰화하는 방식이다.  
    - subword는 하나의 단어를 구성하는 단어들을 말한다.(coworker: co, work, er)
- 주로 자주 등장하는 단어의 일부를 공통된 토큰으로 만들고, 희귀하거나 복합적인 단어는 작은 조각(subword)으로 나누어 처리한다.
- 단어 자체를 그대로 사용하기보다는 단어의 일부를 나누어 처리함으로써 새로운 단어나 미등록 단어(Out-of-Vocabulary) 문제를 줄일 수 있다.

## 장점

1. **미등록 단어 처리 가능**  
   -  새로운 단어(신조어, 속어, 고유어등)가 등장해도 미리 정의된 subword를 조합해서 표현할 수 있어 OOV 문제를 줄일 수 있다.  

2. **어휘 크기 축소**  
   - 같은 subword를 여러 단어에서 공유함으로써, 완전한 단어를 사용하는 경우보다 어휘집의 크기를 작게 유지할 수 있다.


## 종류

1. **Byte-Pair Encoding (BPE)**  
   - 자주 등장하는 문자 쌍을 반복적으로 병합해 서브워드를 생성하는 방식.
   - ex) low, lower, lowest -> 'low'가 많이 나오는구나 ! -> low를 따로 잘라
   - OpenAI의 GPT 모델에 사용된 토크나이저이다.

2. **Unigram**  
   - 빈도기반 확률모델에 따라 subword 단위를 선택하는 방식이다.  
   - BPE보다 유연하여 더 다양한 분할 결과를 얻을 수 있다.

3. **WordPiece**  
   - BPE와 유사하지만, 빈도수가 아니라, 가능성이 높은 조합(합쳐질 가능성이 높은 subword)에 기반해 subword들을 찾는다.
   - Google의 BERT 모델에 사용된 토크나이저이다.

# Byte Pair Encoding 방식 - 가장 많이 나오는 서브워드

- 원래 Text data 압축을 위해 만들어진 방법으로 text 에서 많이 등장하는 두글자 쌍의 조합을 찾아 부호화하는 알고리즘이다. 
- 연속된 글자 쌍이 더 나타나지 않거나 정해진 어휘사전 크기에 도달 할 때 까지 조합을 찾아 부호화 하는 작업을 반복한다.

## text 압축 방식의 예
- 원문: abracadabra
1. AracadAra: ab -> A :=> 원문에서 가장 빈도수 많은 ab를 A(부호로 아무 글자나 사용할 수 있다.)로 치환
2. ABcadAB: ra -> B :=> 1에서 가장 빈도수가 많은 ra를 B로 치환
3. CcadC: AB -> C :=> 2에서 가장 빈도수 맣은 AB를 C로 치환한다.(치환된 글자 쌍도 변환대상에 포함된다.)

## BPE Tokenizer 방식
- BPE 토크나이저는 자주 등장하는 글자 쌍을 찾아 치환하는 대신 **단어 사전**에 추가한다.
- 각 '글자'를 저장하고 빈도수가 높은 서브워드를 추가하는 방식

### 예)
1. 말뭉치의 토큰들의 빈도수, 어휘사전은 아래와 같을 경우
    - 빈도사전: ('low', 5), ('lower', 2), ('newest', 6), ('widest', 3)
    - 어휘사전: ['low', 'lower', 'newest', 'widest']
2. 빈도 사전내의 모든 단어들을 글자 단위로 나눈다. (Pre Tokenization)
    - 빈도사전: ('l', 'o', 'w',  5), ('l', 'o', 'w', 'e', 'r', 2), ('n', 'e', 'w', 'e', 's', 't', 6), ('w', 'i', 'd', 'e', 's', 't', 3)
    - 어휘사전: ['d', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w']
3. 빈도 사전을 기준으로 가장 자주 등장하는 글자 쌍(byte pair)를 찾는다.  위에서는 **'e'와 's'가 총 9번으로 가장 많이 등장함**. 'e'와 's'를 'es'로 합치고 어휘 사전에 추가한다. -> 기존 'e', 's'는 이미 사전에 있으므로 병합하는거야
    - 빈도사전: ('l', 'o', 'w',  5), ('l', 'o', 'w', 'e', 'r', 2), ('n', 'e', 'w', **'es'**, 't', 6), ('w', 'i', 'd', **'es'**, 't', 3)
    - 어휘사전: ['d', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w', **'es'**]
4. 3 번의 과정을 계속 반복한다. 빈도수가 가장 많은 'es'와 't' 쌍을 'est'로 병합하고 'est'를 어휘 사전에 추가한다.
    - 빈도사전: ('l', 'o', 'w',  5), ('l', 'o', 'w', 'e', 'r', 2), ('n', 'e', 'w', **'est'**, 6), ('w', 'i', 'd', **'est'**, 3)
    - 어휘사전: ['d', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w', **'es'**, **'est'**]
5. 만약 10번 반복했다고 하면 다음과 같은 빈도 사전과 어휘 사전이 생성된다.
    - 빈도 사전: (**'low'**, 5), (**'low'**, 'e', 'r', 2), ('n', 'e', 'w', **'est'**, 6), ('w', 'i', 'd', **'est'**, 3)
    - 어휘사전: ['d', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w', **'es'**, **'est'**, **'lo'**,**'low'**, **'low'**, **'ne'**, **'new'**, **'newest'**, **'wi'**, **'wid'**, **'widest'**]

- 위와 같이 어휘 사전이 만들어 지면 원래 어휘서전에 없던 것들에 대한 처리를 할 수있다.
    - ex)
        - 'newer' :=> 'new', 'e', 'r', 
        - 'lowest' :=> 'low', 'est'
        - 'wider' :=> 'wid', 'e', 'r'

# WordPiece tokenizer - 가장 나올법한 서브워드

- Byte Pair Encoding 이 빈도 기반이라면 wordpiece tokenizer는 확률 기반으로 글자 쌍을 병합한다.
    - 이를 통해 '의미있는 조합'의 특징을 가져갈 수 있다.
- 두개 글자 쌍의 빈도수를 각 개별 글자 빈도수의 곱으로 나눈 점수가 가장 높은 순서대로 글자쌍을 묶어 나간다.

$$
score = \cfrac{f(x, y)}{f(x)\cdot f(y)} 
$$

함수 f는 빈도를 나타내며 x, y는 병합하려는 하위 단어이다.

- 빈도사전: ('l','o','w', 5), ('l','o','w', 'e', 'r', 2), ('n', 'e', 'w', 'e', 's', 't', 6), ('w', 'i', 'd', 'e', 's', 't', 3)
- 어휘사전: ('d', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w')
- 가장 빈도수가 높은 쌍은 'e','s'로 9번 등장한다. 이때 각 글자는 전체에서 각각 'e'는 17번, 's'는 9번 등장한다. 위 공식에 대입하면 score는 $\frac{9}{17 \times 9} \approx 0.06$ 이다.
- 'i'와 'd' 쌍은 3번만 등장하지만 전체에서 각각 'i' 3번, 'd' 3번 등장한다. 그래서 score는 $\frac{3}{3 \times 3} \approx 0.33$ 이다.
- 나타난 빈도수는 'es' 가 많치만 더 높은 score를 가지는 'id' 쌍을 병합한다.
- 빈도사전: ('l','o','w', 5), ('l','o','w', 'e', 'r', 2), ('n', 'e', 'w', 'e', 's', 't', 6), ('w', **'id'**, 'e', 's', 't', 3)
- 어휘사전: ('d', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w', **'id'**)
위의 작업을 반복해 연속된 글자 쌍이 더이상 나타나지 않거나 어휘 사전 max 크기에 도달할 때 까지 학습한다.

# Unigram 방식 - 가장 그럴법한 서브워드 세트
- 빈도 기반 확률 모델을 사용하여 효율적으로 서브워드를 선택하고, 불필요한 서브워드를 제거해 최적의 어휘 크기를 찾는 알고리즘
- 


- **초기 어휘 집합 구성**
    - 대상 text에 모든 단어와 그 서브스트링을 포함한 어휘 집합을 생성한다. 이 어휘 집합은 나올 수있는 모든 subword들을 다 모아놓은 것이다. 
    - 예를 들어 "hug" 단어의  ["h", "u", "g", "hu", "ug", "hug"]  substring을 만든다. 이들이 subword 후보가 된다.
- **각 Subword의 빈도수 기반 확률 계산**
    -  $\cfrac{subword가\;나타난\;횟수}{전체\;빈도수}$ 로 각 subword들의 나타난 확률을 계산한다.
- **가능한 분할에 대한 확률 계산**
    - 단어를 여러 서브워드로 분할할 수 있는 경우, 각 분할에 대한 전체 확률을 계산한다.
    - 확률 계산은 $ P(subword1)\;\times \; P(subword2)\;\times\; ..$ 으로 계산한다.
    - 예를 들어 "hug" 를 분할 한다고 했을 때
        1. \["h", "u", "g"\]: $ P(h) \times P(u) \times P(g) $
        2. \["hu", "g"\]: $ P(pu) \times P(g) $

   - 각각의 확률을 계산한 후, **가장 높은 확률**을 가진 분할을 선택한다.
     - 위 예에서 만약 1의 확률이 0.01 이고 2의 확률이 0.00001 이라면 첫번째 분할이 선택된다.

- **서브워드 제거**
    - 위의 훈견과정에서 불필요한 서브워드를 제거하면서 최적의 어휘 집합을 찾아간다. 
    - 제거 대상은 빈도수가 낮거나 조합에 크게 영향을 주지 않은 subword들이다.

In [21]:
from Korpora import Korpora

corpus = Korpora.load('korean_petitions')


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : Hyunjoong Kim lovit@github
    Repository : https://github.com/lovit/petitions_archive
    References :

    청와대 국민청원 게시판의 데이터를 월별로 수집한 것입니다.
    청원은 게시판에 글을 올린 뒤, 한달 간 청원이 진행됩니다.
    수집되는 데이터는 청원종료가 된 이후의 데이터이며, 청원 내 댓글은 수집되지 않습니다.
    단 청원의 동의 개수는 수집됩니다.
    자세한 내용은 위의 repository를 참고하세요.

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/

[Korpora] Corpus `korean_petitions` is already installed at C:\Users\jinhy\Korpora\korean_petitions\petitions_2017-08
[Korpora] Corpus `korean_petitions` is already installed at C:\Users\jinhy\Korpora\korean_petitions\petitions_2017-09
[Korpora] Corpus `korean_petitions` is already installed at 

In [22]:
petitions = corpus.get_all_texts()  # 청원 데이터를 반환. list[str] str : 개별 청원 
len(petitions)

433631

In [23]:
# 파일로 저장
save_petitions_path = 'data/petitions_corpus.txt'
with open(save_petitions_path, 'wt', encoding='utf-8') as fw:
    # fw.writelines(petitions)        # 이러면 근데 줄바꿈이 안되기는 해
    for p in petitions:
        fw.write(p+'\n')
        

In [24]:
# 저장된 txt 읽기
with open(save_petitions_path, 'rt') as fr:
    petitions_txt = fr.read()

# Hugging Face의 tokenizers 패키지 사용해 토큰화 수행
- Subword tokenization을 처리하는 다양한 패키지(라이브러리)가 있다.

## 주요 라이브러리 
- [tokenizers](https://huggingface.co/docs/tokenizers/index)
    - huggingface에서 개발한 tokenizer 라이브러리로 BPE, WordPiece, Unigram 알고리즘을 지원한다. 
    - 설치: `pip install tokenizers`
- [Sentencepiece](https://github.com/google/sentencepiece)
    - 구글에서 개발한 subword tokenizer 라이브러리로 BPE, Unigram 방식 지원.
    - 설치: `pip install sentencepiece`

> ### korpora 말뭉치
> - 다양한 한글 데이터셋을 제공하는 패키지
> - `pip install korpora`

## Hugging Face tokenizers 패키지이용

- 서브워드의 경우 그냥 토큰화와 다르게 문맥을 읽어야하므로 별도의 학습이 요구되는거야
- 설치: `pip install tokenizers`
- Tokenizer 생성
    - 토큰화 알고리즘을 지정해 instance 생성.
- Trainer 생성
    - 학습 파라미터를 설정해서 instance 생성
- Tokenizer 학습
    - train() 메소드: 학습 text 파일 경로를 지정해서 학습
    - train_from_iterator() 메소드: 학습할 string들을 iterator를 통해 제공.
- https://github.com/huggingface/tokenizers



In [25]:
import time
from tokenizers import Tokenizer
from tokenizers.models import BPE #  WordPiece, Unigram
# subword 알고리즘을 적용하기 전에 어떻게 나눠놓을 것인지. -> 공백을 기준으로 처음 토큰을 나누어라 
# nltk의 tokenizer과 동일한 역할을 하기위해 불러온거야 
from tokenizers.pre_tokenizers import Whitespace
# Trainer (학습)
from tokenizers.trainers import BpeTrainer

In [26]:
# Tokenizer 객체 생성
# bpe = BPE()
# type(bpe), type(tokenizer) - (tokenizers.models.BPE, tokenizers.Tokenizer)

## subword 알고리즘을 구현한 Tokenizer의 객체를 넣어 생성. (BPE)
## unk_token : OOV(Unknown) 단어(토큰)을 처리할 토큰을 지정.
tokenizer = Tokenizer(BPE(unk_token='[UNK]'))
# pre_tokenizer 를 등록              -> __init__()에서 기본값으로 안 받아줘..... 
tokenizer.pre_tokenizer = Whitespace()
# tokenizer를 학습시키는 Trainer 객체 -> Tokenizer 알고리즘별로 당연히 Trainer 알고리즘이 다르겠죠?
# initializer에 어떻게 학습시킬지 설정
trainer = BpeTrainer(
    vocab_size=10_000,          # 어휘 사전의 최대 크기(고유 토큰의 최대 개수) - 보통 30_000 개
    min_frequency=10,           # 사전에 넣을 토큰의 **최소 출현 횟수**(빈도수 - BPE)
    special_tokens=['[UNK]', '[PAD]']   # 어휘 사전에 추가할 특수(목적) 토큰  - 여기서 만드는거고 BPE 객체는 이걸 쓰는거야 
)
tokenizer
# special token: [UNK] - OOV 토큰을 표시.
#                [PAD] - 문장의 토큰수를 맞추기 위한 padding
#                [CLS] - 문장의 시작을 표시 + 전체 문장의 의미를 저장하는 토큰을 사용(BERT 모델)
#                [SOS] - 문서의 시작 / [EOS] - 문서의 끝
#                [SEP] - 문서 속 여러 문장 구분 ex) 서로 다른 개념, 질문과 응답 등의 stream?을 구분하기 위함 
#                [MASK] - 일부 토큰을 가리는 토큰, 모르는 단어, 추측 시 유용하게 사용 


Tokenizer(version="1.0", truncation=None, padding=None, added_tokens=[], normalizer=None, pre_tokenizer=Whitespace(), post_processor=None, decoder=None, model=BPE(dropout=None, unk_token="[UNK]", continuing_subword_prefix=None, end_of_word_suffix=None, fuse_unk=False, byte_fallback=False, ignore_merges=False, vocab={}, merges=[]))

In [27]:
# 학습 - corpus를 받아서 서브단어 어휘사전을 만드는 작업
s = time.time()
tokenizer.train(['data/petitions_corpus.txt'], trainer=trainer)
e = time.time()
print(f'걸린 시간 : {e-s} 초')

걸린 시간 : 88.37283396720886 초


In [29]:
# 학습된 tokenizer를 disk에 저장
import os
os.makedirs('saved_models', exist_ok=True)

# 저장 - tokenizer.save(경로)
save_path = 'saved_models/petitions_bpe.json'
tokenizer.save(save_path)

In [30]:
# 저장된 모델 load
load_tokenizer = Tokenizer.from_file(save_path)

In [31]:
# Vocab Size
print('vocab size(어휘 사전 어휘 수) : ', tokenizer.get_vocab_size())

vocab size(어휘 사전 어휘 수) :  10000


In [33]:
# 어휘 사전
tokenizer.get_vocab()

{'죗': 6560,
 '贊': 3461,
 '튠': 7240,
 '뼌': 5634,
 '해보': 9340,
 '않도록': 9578,
 '深': 2574,
 '뒺': 4741,
 '뢸': 5053,
 '방해': 9829,
 '漏': 2618,
 '推': 2196,
 '陵': 3714,
 '석': 5746,
 'ญ': 322,
 '쩠': 6713,
 '点': 2659,
 '㈏': 1115,
 '수준': 8844,
 '행을': 9321,
 '固': 1637,
 '般': 3170,
 '🔴': 8067,
 '꼻': 4239,
 'ㅛ': 1093,
 '連': 3562,
 '입니다': 8212,
 '⒋': 639,
 '履': 1872,
 '提': 2200,
 '봍': 5471,
 '비스': 9251,
 '岳': 1880,
 '꽥': 4257,
 '⇆': 542,
 '庇': 1944,
 '拳': 2172,
 '왝': 6251,
 '쭈': 6748,
 '剽': 1455,
 '기는': 8660,
 'а': 242,
 '삻': 5705,
 '侈': 1293,
 '阿': 3699,
 '세금을': 9270,
 '툭': 7216,
 '했지만': 9516,
 '없어서': 9553,
 '途': 3556,
 '핦': 7433,
 '제를': 8848,
 '샬': 5736,
 '王': 2735,
 '二': 1215,
 '坑': 1658,
 '猛': 2721,
 '가서': 8766,
 '쑫': 6028,
 '처벌을': 9245,
 '늪': 4539,
 '摠': 2214,
 '联': 3105,
 '쎵': 5996,
 '됍': 4688,
 '💵': 8052,
 '100': 8603,
 '같은': 8279,
 '廟': 1964,
 '풰': 7382,
 '\ue470': 7675,
 '☹': 751,
 '仅': 1241,
 '놁': 4458,
 '뮌': 5308,
 '翰': 3093,
 '휵': 7621,
 '거리': 9050,
 '뮬': 5317,
 '앍': 6078,
 '여러분': 9291,
 '

In [34]:
# 테스트 문장
sports_txt = "프리미어리그 역대 개인 최다골 기록을 보유하고 있는 시어러가 손흥민의 골 결정력을 재차 극찬했다."
petition_txt = "이 글을 쓴 이유는 다름아닌 '전안법'시행 반대를 주장하기 위해서입니다. 먼저, '전안법'은 전기용품 및 생활용품을 판매하는 업체에서 KC인증마크를 의무적으로 받는 것입니다."
comment_txt = "멋진 식사를 즐기기에 좋은 장소 - 채식 메뉴가 정말 훌륭했습니다. 당근 케이크는 아마도 내가 먹어본 디저트 중 최고였을 거예요."

In [35]:
# text -> token 화
token_output = tokenizer.encode(sports_txt)
token_output

Encoding(num_tokens=34, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [36]:
# token -> word
len(token_output.tokens), token_output.tokens

(34,
 ['프',
  '리',
  '미',
  '어',
  '리',
  '그',
  '역',
  '대',
  '개인',
  '최',
  '다',
  '골',
  '기록',
  '을',
  '보유',
  '하고',
  '있는',
  '시',
  '어',
  '러',
  '가',
  '손',
  '흥',
  '민',
  '의',
  '골',
  '결정',
  '력을',
  '재',
  '차',
  '극',
  '찬',
  '했다',
  '.'])

In [37]:
# token -> token id
token_output.ids

[7390,
 5123,
 5330,
 6140,
 5123,
 4128,
 6180,
 4589,
 8414,
 6891,
 4563,
 4027,
 9449,
 6364,
 9442,
 8209,
 8219,
 5908,
 6140,
 4980,
 3902,
 5802,
 7638,
 5332,
 6384,
 4027,
 8940,
 8644,
 6449,
 6793,
 4129,
 6795,
 8565,
 15]

In [38]:
output = tokenizer.encode(petition_txt)
print(len(output.tokens), output.tokens)
output2 = tokenizer.encode(comment_txt)
print(len(output2.tokens), output2.tokens)

52 ['이', '글을', '쓴', '이유는', '다', '름', '아닌', "'", '전', '안', '법', "'", '시행', '반대', '를', '주장', '하기', '위해서', '입니다', '.', '먼저', ',', "'", '전', '안', '법', "'", '은', '전기', '용', '품', '및', '생활', '용', '품', '을', '판매', '하는', '업체', '에서', 'K', 'C', '인', '증', '마', '크', '를', '의무', '적으로', '받는', '것입니다', '.']
45 ['멋', '진', '식', '사를', '즐', '기', '기에', '좋은', '장', '소', '-', '채', '식', '메', '뉴', '가', '정말', '훌', '륭', '했습니다', '.', '당', '근', '케', '이', '크', '는', '아', '마', '도', '내가', '먹', '어', '본', '디', '저', '트', '중', '최고', '였', '을', '거', '예', '요', '.']


In [39]:
# offsets[토근 리스트의 index]
output.offsets[3]

(7, 10)

In [40]:
# 토큰 문자열 -> 토큰 id
print(tokenizer.token_to_id('이유'))
# 토큰 id -> 토큰 문자열
print(tokenizer.id_to_token(8341))

8341
이유


In [44]:
ids = [100, 8231, 8341, 20, 700, 1238, 8465, 8345, 8798]
decode_output = tokenizer.decode(ids)
print(decode_output)
print(tokenizer.decode(output.ids))

¥ 니까 이유 3 ▶ 什 병원 ?? 구요
이 글을 쓴 이유는 다 름 아닌 ' 전 안 법 ' 시행 반대 를 주장 하기 위해서 입니다 . 먼저 , ' 전 안 법 ' 은 전기 용 품 및 생활 용 품 을 판매 하는 업체 에서 K C 인 증 마 크 를 의무 적으로 받는 것입니다 .


# WordPiece 모델 학습

In [46]:
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import WordPieceTrainer

# Tokenizer 객체 생성
tokenizer2 = Tokenizer(
    WordPiece(unk_token='[UNK]')
)

# Pre_Tokenizer
tokenizer2.pre_tokenizer = Whitespace()

# Trainer
trainer2 = WordPieceTrainer(
    vocab_size=20_000,
    min_frequency=10,
    special_tokens=['[UNK]', '[PAD]', '[SEP]', '[EOS]', '[SOS]']
)


In [50]:
# 학습시키기
import time
s = time.time()
tokenizer2.train(['data/petitions_corpus.txt'], trainer=trainer2)
e = time.time()
print(f'{e-s}초 걸림')

208.18228316307068초 걸림


In [51]:
# 저장하기
tokenizer2.save('saved_models/petitions_wordpiece.json')

In [ ]:
# 불러오기
load_tokenizer2 = Tokenizer.from_file('saved_models/petitions_wordpiece.json')

In [52]:
tokenizer2.get_vocab_size()

20000

In [ ]:
output = tokenizer2.encode(comment_txt)
output, output.tokens[:10]
# '##' -> 뒷부분에 붙는 단어 -> 서브워드로 합칠때 편리해

(Encoding(num_tokens=44, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
 ['멋', '##진', '식', '##사를', '즐', '##기', '##기에', '좋은', '장', '##소'])

# Unigram 모델 학습

In [61]:
from tokenizers import Tokenizer
from tokenizers.models import Unigram
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import UnigramTrainer

# Tokenizer 객체 생성
tokenizer3 = Tokenizer(
    Unigram()
)

# Pre_Tokenizer
tokenizer3.pre_tokenizer = Whitespace()

# Trainer
trainer3 = UnigramTrainer(
    vocab_size=10_000,
    min_frequency=10,
    special_tokens=['[UNK]', '[PAD]', '[SEP]', '[EOS]', '[SOS]']
)


In [62]:
# 학습시키기
import time
s = time.time()
tokenizer3.train(['data/petitions_corpus.txt'], trainer=trainer3)
e = time.time()
print(f'{e-s}초 걸림')

421.48179483413696초 걸림


In [64]:
# 저장하기
tokenizer3.save('saved_models/petitions_unigram.json')

In [65]:
tokenizer3.token_to_id('안녕하세요')

840

In [66]:
tokenizer3.id_to_token(930)

'지켜'

In [67]:
output = tokenizer3.encode(comment_txt)
output.tokens[:10]

['멋', '진', '식', '사', '를', '즐', '기', '기에', '좋은', '장']

In [70]:
output.ids[:10]

[2663, 92, 165, 39, 13, 2232, 28, 713, 485, 63]

In [71]:
tokenizer3.decode(output.ids)

'멋 진 식 사 를 즐 기 기에 좋은 장 소 - 채 식 메 뉴 가 정말 훌 륭 했습니다 . 당 근 케 이 크 는 아 마 도 내가 먹 어 본 디 저 트 중 최 고 였 을 거 예 요 .'

In [ ]:
# Wordpiece의 경우 ##을 이용하는데 이를 통해 본래의 문장을 원복하기가 쉽다.
output2 = tokenizer2.encode(comment_txt)
tokenizer2.decode(output2.ids)

'멋 ##진 식 ##사를 즐 ##기 ##기에 좋은 장 ##소 - 채 ##식 메 ##뉴 ##가 정말 훌 ##륭 ##했습니다 . 당 ##근 케 ##이 ##크 ##는 아마 ##도 내가 먹 ##어 ##본 디 ##저 ##트 중 최고 ##였 ##을 거 ##예 ##요 .'